In [1]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.interpolate import interp1d, CubicSpline
from scipy.integrate import cumtrapz
from astropy.cosmology import Planck15 as cosmo
from classy import Class
from astropy.cosmology import FlatLambdaCDM
import dill
import os
import json

### 1. Cosmology & Configuration

In [2]:
# Define basic cosmological parameters
h = cosmo.h
Omega_b = cosmo.Ob0
Omega_cdm = cosmo.Odm0
Omega_m = Omega_b + Omega_cdm
Omega_Lambda = 1.0 - Omega_m

omega_b = cosmo.Ob0*h*h
omega_cdm = cosmo.Odm0*h*h

In [3]:
# Define the redshift grid for integrations
z_min = 0.0
z_max = 10.0
z_step = 0.0001
zz_grid = np.arange(z_min, z_max, z_step)

# Define the observational range
start_range = 0.5
end_range = 3.5

# Define multipole scale
lmax = int(200)
ell = np.arange(lmax+1)[2:]

In [4]:
# Users can toggle between 'ET2LCE', 'ET2L', and 'ET_DELTA' here.
active_network = 'ET2LCE'

gw_config = {
    'ET2LCE': {
        'A': 3.3613, 'b': 3.0532, 'c': 2.1555, 
        'detected_per_year': 19299. / 0.2,
        'sigma_gw_2snr': np.array([0.027, 0.036, 0.041, 0.046, 0.053, 0.056, 0.060, 0.064, 0.072, 0.082]),
        'gw_damping': np.array([52441, 29241, 22201, 17161, 12996, 11236, 9025, 7569, 6084, 4761])
    },
    'ET2L': {
        'A': 4.2436, 'b': 3.1699, 'c': 2.3106, 
        'detected_per_year': 17121. / 0.2,
        'sigma_gw_2snr': np.array([0.046, 0.060, 0.071, 0.079, 0.089, 0.095, 0.098, 0.107, 0.116, 0.124]),
        'gw_damping': np.array([400, 256, 256, 225, 196, 169, 169, 169, 144, 144])        
    },
    'ET_DELTA': {
        'A': 5.0610, 'b': 3.2763, 'c': 2.4360, 
        'detected_per_year': 16317. / 0.2,
        'sigma_gw_2snr': np.array([0.058, 0.076, 0.087, 0.095, 0.105, 0.110, 0.115, 0.121, 0.127, 0.135]),
        'gw_damping': np.array([49, 36, 49, 36, 49, 36, 36, 36, 36, 36])
    }
}

### 2. Population Functions

In [5]:
def get_gw_dndz(z: np.ndarray, network: str) -> np.ndarray:
    """Computes the GW number density distribution dN/dz for a given network."""
    cfg = gw_config[network]
    return cfg['detected_per_year'] * cfg['A'] * (z**cfg['b']) * np.exp(-cfg['c'] * z)


def get_hi_dndz(z: np.ndarray) -> np.ndarray:
    """Computes the HI temperature brightness distribution."""
    E_z = (cosmo.H(z) / (100 * cosmo.h)).value 
    Omega_hi = 4 * ((1 + z)**0.6) * 1e-4 
    return 44e-6 * (Omega_hi * cosmo.h / 2.45 / 1e-4) * (((1 + z)**2) / E_z)


def get_hi_rho(z: np.ndarray) -> np.ndarray:
    """Computes the physical density evolution of HI."""
    rho_crit0 = cosmo.critical_density0.value
    return 4 * ((1 + z)**0.6) * 1e-4 * rho_crit0


def get_magnification_bias_gw(z: np.ndarray) -> np.ndarray:
    """
    Returns the magnification bias s_GW(z) based on a custom interpolated model.
    Saturates smoothly to 0.40 at z >= 7.
    """
    z_nodes = np.array([0.0, 1.0,  2.0,  3.0,  5.0,  7.0])
    s_nodes = np.array([0.0, 0.08, 0.20, 0.30, 0.37, 0.40])
    
    spline_model = CubicSpline(z_nodes, s_nodes, bc_type=((2, 0.0), (1, 0.0)))
    
    # Evaluate spline and clamp values for z > 7
    s_gw = spline_model(z)
    #if isinstance(s_gw, np.ndarray):
    #    s_gw[z > 7.0] = 0.40
    return s_gw


def get_bias_gw(z: np.ndarray) -> np.ndarray:
    """Astrophysical clustering bias for GW sources."""
    return (0.948 * np.exp(-0.553 * (z**1.034))) + (z**0.996)


def get_bias_hi(z: np.ndarray) -> np.ndarray:
    """Astrophysical clustering bias for HI sources."""
    return (0.22 * ((1 + z)**1.47)) + 0.63

### 3. Generating the Grid and Binning Data

In [6]:
# 1. Compute and normalize the global theoretical distributions
dn_gw = get_gw_dndz(zz_grid, active_network)

total_gw_sources = int(np.trapz(dn_gw, zz_grid))
print(f"Total GW Sources: {total_gw_sources:.0f}")

dn_gw /= np.trapz(dn_gw, zz_grid)  # Normalize to 1

n_gw = cumtrapz(dn_gw, zz_grid, initial=0)

dn_hi = get_hi_dndz(zz_grid)
rho_hi = get_hi_rho(zz_grid)

Total GW Sources: 92559


In [7]:
# 2. Extract GW sources within the observational window (0.5 < z < 3.5)
mask_window = (zz_grid >= start_range) & (zz_grid <= end_range)
z_window = zz_grid[mask_window]
dn_window = dn_gw[mask_window]

In [8]:
# 3. Calculate equi-populated GW bins based on the CDF
n_z_window = cumtrapz(dn_window, z_window, initial=0)
cdf_window = n_z_window / n_z_window[-1]

inverse_cdf_gw = interp1d(cdf_window, z_window, kind='linear', bounds_error=False, fill_value="extrapolate")

number_sameN_bins_gw = 10
target_fractions = np.linspace(0, 1, number_sameN_bins_gw + 1) # 10 bins requires 11 edges
zz_edges_eq_gw = inverse_cdf_gw(target_fractions)
zz_edges_eq_gw[0] = start_range
zz_edges_eq_gw[-1] = end_range

zz_gw = (zz_edges_eq_gw[:-1] + zz_edges_eq_gw[1:]) / 2.0
num_zbins_gw = len(zz_gw) 

In [9]:
counts_per_bin = []

for i in range(len(zz_edges_eq_gw) - 1):
    z_lower = zz_edges_eq_gw[i]
    z_upper = zz_edges_eq_gw[i+1]
    
    # Integrate the normalized PDF within this bin
    mask = (zz_grid >= z_lower) & (zz_grid <= z_upper)
    
    # Fraction of total sources falling in this bin
    fraction = np.trapz(dn_gw[mask], zz_grid[mask])

    # Absolute count
    N_in_bin = fraction * total_gw_sources
    counts_per_bin.append(N_in_bin)

mean_n_per_bin = np.array(counts_per_bin)

print("Corrected Counts per bin (Should be constant):")
print(mean_n_per_bin)

Corrected Counts per bin (Should be constant):
[8488.63355414 8487.93857638 8485.9318611  8487.41394672 8485.43541636
 8490.72330132 8485.34517561 8488.76940168 8488.89210363 8490.06667991]


In [10]:
# 4. Define uniform HI bins
width_rangehi = 0.1
zz_edges_hi = np.arange(start_range, end_range + width_rangehi, width_rangehi)
zz_hi = 0.5 * (zz_edges_hi[:-1] + zz_edges_hi[1:])
num_zbins_hi = len(zz_hi)

In [11]:
# 5. Calculate bin widths
width_hi = np.full(len(zz_hi), width_rangehi)
width_gw = gw_config[active_network]['sigma_gw_2snr'] * zz_gw

# 6. Calculate Biases
bias_hi = get_bias_hi(zz_hi)
bias_gw = get_bias_gw(zz_gw)

magnBias_hi = np.full(len(zz_hi), 0.4)
magnBias_gw = get_magnification_bias_gw(zz_gw)

### 4. Combine, Sort, and Format for Configuration Files

In [12]:
# 1. Concatenate the HI and GW arrays into unified arrays
zz_combined = np.concatenate((zz_hi, zz_gw))
width_combined = np.concatenate((width_hi, width_gw))
bias_combined = np.concatenate((bias_hi, bias_gw))

num_zbins_tot = len(zz_combined) 

In [13]:
# 2. Sort EVERYTHING by redshift 
# (MultiCLASS and Fisher codes will crash if z-bins are not strictly ascending!)
idx_sort = np.argsort(zz_combined)

z_sorted = zz_combined[idx_sort]
width_sorted = width_combined[idx_sort]

In [14]:
# 3. Computed full sorted bias arrays
bias_sorted = []
magnBias_sorted = []

for i in range(len(z_sorted)):
    bias_sorted.append( get_bias_hi(z_sorted[i]) )
for i in range(len(z_sorted)):
    bias_sorted.append( get_bias_gw(z_sorted[i]) )

for i in range(len(z_sorted)):
    magnBias_sorted.append( 0.4 )
for i in range(len(z_sorted)):
    magnBias_sorted.append( get_magnification_bias_gw(z_sorted[i]) )

In [15]:
# 4. Set up the strings for later use
zz_string = ", ".join(f"{x:.4f}" for x in zz_combined)
zz_string_hi = ", ".join(f"{x:.4f}" for x in zz_hi)
zz_string_gw = ", ".join(f"{x:.4f}" for x in zz_gw)
z_string_sorted = ", ".join(f"{x:.4f}" for x in z_sorted)

width_string_hi = ", ".join(f"{x:.4f}" for x in width_hi)
width_string_gw = ", ".join(f"{x:.4f}" for x in width_gw)
width_string = ", ".join(f"{x:.4f}" for x in width_combined)
width_string_sorted = ", ".join(f"{x:.4f}" for x in width_sorted)

selectBiashi_string = ", ".join(f"{x:.4f}" for x in bias_hi)
selectBiasgw_string = ", ".join(f"{x:.4f}" for x in bias_gw)
selectBias_string = ", ".join(f"{x:.4f}" for x in bias_sorted)

magnBiashi_string = ", ".join(f"{x:.4f}" for x in magnBias_hi)
magnBiasgw_string = ", ".join(f"{x:.4f}" for x in magnBias_gw)
magnBias_string = ", ".join(f"{x:.4f}" for x in magnBias_sorted)

### 5. Export Population Data to Text Files

In [16]:
# Save GW Distribution
output_gw = np.column_stack((zz_grid, dn_gw))
np.savetxt('selection_gw.txt', output_gw, fmt="%.6f")

# Save HI Distribution
output_hi = np.column_stack((zz_grid, dn_hi))
np.savetxt("selection_hi.txt", output_hi, fmt="%.6f") 

# Save HI Evolution
output_rho_hi = np.column_stack((zz_grid, rho_hi))
np.savetxt("evo_hi.txt", output_rho_hi, fmt="%.6f")

print("Successfully generated and saved theoretical distributions.")

Successfully generated and saved theoretical distributions.


### 6. Compute Angular Powe Spectra

In [17]:
# Define the global precision parameters shared by all CLASS runs
base_class_params = {
    'output': 'nCl, mPk',
    'number count contributions': 'density',

    'modes': 's',              
    'h': h,
    'omega_b': omega_b,
    'omega_cdm': omega_cdm,
    'l_max_lss': lmax,
    'lensing': 'no',
    'non linear': 'none',
    'P_k_max_h/Mpc': 50,       
    'z_max_pk': 8,
    'recombination': 'HyREC',
    'l_linstep': 5,
    'tol_perturb_integration': 1e-7,
    'perturb_sampling_stepsize': 0.01,
    'k_per_decade_for_pk': 70,      
    'k_max_tau0_over_l_max': 3.0,   

    'z_reio': 7.6711,
    'N_ur': 3.044,
    'A_s': 2.100549e-9,
    'n_s': 0.9660499,
    
    'l_switch_limber_for_nc_local_over_z': 60
}

In [18]:
# Define function for CLASS runs
def run_class(tracer_type: str, z_str: str, width_str: str, bias_str: str, magn_str: str, 
              file1: str, file2: str = None) -> np.ndarray:
    """
    Initializes MultiCLASS, sets parameters based on the tracer type (HI-HI, GW-GW, or HI-GW),
    computes the power spectra, extracts the density C_l array, and cleans up memory.
    """
    params = base_class_params.copy()
    
    # 1. Selection & Multitracing Settings
    if tracer_type == 'HIxHI':
        num_bins = num_zbins_hi
    elif tracer_type == 'GWxGW':
        num_bins = num_zbins_gw
    elif tracer_type == 'GWxHI':
        num_bins = num_zbins_tot
    else:
        print('No correct tracer_type was passed')

    is_multitracing = 'yes' if tracer_type == 'GWxHI' else 'no'

    params.update({
        'selection_multitracing': is_multitracing,

        'selection_window': 'gaussian',
        'selection_mean': z_str, 
        'selection_width': width_str, 
        'selection_bias': bias_str,
        'selection_magnification_bias': magn_str,

        'selection_dNdz_1': 'file',
        'selection_dNdz_filepath_1': file1,
        'selection_dNdzevolution_filepath_1': file1 if tracer_type == 'GWxGW' else '/home/matteo/evo_hi.txt',

        'non_diagonal': num_bins - 1
    })

    # Cross-correlation requires two file paths
    if is_multitracing == 'yes':
        params.update({
            'selection_dNdz_2': 'file',
            'selection_dNdz_filepath_2': file2,
            'selection_dNdzevolution_filepath_2': file2
        })

    # 2. Execute CLASS
    print(f"Running MultiCLASS for {tracer_type}...")
    classRun = Class()
    classRun.set(params)
    classRun.compute()

    # 3. Extract and Stack Data
    cl_dict = classRun.density_cl(lmax=lmax)
    N_iter = num_bins * num_bins if is_multitracing == 'yes' else 0.5 * num_bins * (num_bins + 1)

    try:
        cl_stack = np.vstack([cl_dict['dd'][i][2:] for i in range(int(N_iter))])
    except KeyError as e:
        print(f"\nCRITICAL ERROR in {tracer_type}: Expected {N_iter} spectra, but extraction failed at index {e}.")
        print(f"CLASS silently dropped the cross-correlations. Check file paths and parameters.")
        raise e
    
    #cl_stack = np.vstack([cl_dict['dd'][i][2:] for i in range(int(N_iter))])

    # 4. Strict Memory Cleanup (CLASS memory leaks are notoriously bad!)
    classRun.struct_cleanup()
    classRun.empty()
    del classRun
    
    return cl_stack

In [19]:
# Execute the three runs using strings
cl_hh_stack = run_class('HIxHI', zz_string_hi, width_string_hi, 
                        selectBiashi_string, magnBiashi_string, 
                        '/home/matteo/selection_hi.txt')

cl_gg_stack = run_class('GWxGW', zz_string_gw, width_string_gw, 
                        selectBiasgw_string, magnBiasgw_string, 
                        '/home/matteo/selection_gw.txt')

cl_hg_stack = run_class('GWxHI', z_string_sorted , width_string_sorted, 
                        selectBias_string, magnBias_string, 
                        '/home/matteo/selection_hi.txt', '/home/matteo/selection_gw.txt')

Running MultiCLASS for HIxHI...
Running MultiCLASS for GWxGW...
Running MultiCLASS for GWxHI...


### 7. Compute the (Inverted) Covariance Matrix

In [20]:
# 1. Map Redshift Bins back to original order for Cross-Correlation
map_orig_to_sorted = np.argsort(idx_sort)
real_hi_idx = map_orig_to_sorted[:len(zz_hi)]
real_gw_idx = map_orig_to_sorted[len(zz_hi):]

print("Real HI indices in sorted list:", real_hi_idx)
print("Real GW indices in sorted list:", real_gw_idx)

final_cross_stack = []
labels = [] # Optional: just to verify the order

for h, idx_hi in enumerate(real_hi_idx):
    for g, idx_gw in enumerate(real_gw_idx):
        # Calculate the row in the raw stack
        raw_stack_idx = idx_hi * len(zz_combined) + idx_gw
        # Extract and Append
        final_cross_stack.append(cl_hg_stack[raw_stack_idx])
        # Store label for checking (Now it will print correctly!)
        labels.append(f"HI{h+1}xGW{g+1}")

cl_cross_final = np.vstack(final_cross_stack)
bold_cl_vstack = np.concatenate((cl_hh_stack, cl_cross_final, cl_gg_stack))

print(f"Done. Final stack shape: {cl_cross_final.shape}")
print("Order check (first 6):", labels[:6])

Real HI indices in sorted list: [ 0  1  3  4  5  7  8 10 11 13 14 16 17 19 20 22 23 24 26 27 28 30 31 32
 33 34 35 37 38 39]
Real GW indices in sorted list: [ 2  6  9 12 15 18 21 25 29 36]
Done. Final stack shape: (300, 199)
Order check (first 6): ['HI1xGW1', 'HI1xGW2', 'HI1xGW3', 'HI1xGW4', 'HI1xGW5', 'HI1xGW6']


In [21]:
# Initialize tracking arrays and the total matrix
matr_tot = np.empty((num_zbins_tot, num_zbins_tot), dtype=object)
tracer1, tracer2, zz_bin1, zz_bin2 = [], [], [], []

# Map HIxHI block
idx = 0
for i in range(num_zbins_hi):
    for j in range(i, num_zbins_hi):
        tracer1.append(0)
        tracer2.append(0)
        zz_bin1.append(i)
        zz_bin2.append(j)
        matr_tot[i, j] = cl_hh_stack[idx]
        idx += 1

In [22]:
# Map GWxHI cross block
idx = 0
for i in range(num_zbins_hi):
    for j in range(num_zbins_gw):
        tracer1.append( 0 ) 
        tracer2.append( 1 )
        zz_bin1.append( i )
        zz_bin2.append( j )
        matr_tot[i,j+num_zbins_hi] = cl_cross_final[idx]
        idx += 1

In [23]:
# Map GWxGW block
idx = 0
for i in range(num_zbins_gw):
    for j in range(i, num_zbins_gw):
        tracer1.append( 1 )
        tracer2.append( 1 ) 
        zz_bin1.append( i )
        zz_bin2.append( j )
        matr_tot[i + num_zbins_hi, j + num_zbins_hi] = cl_gg_stack[idx]
        idx += 1

In [24]:
# 3. Calculate Knox Covariance 
f_sky = 0.5 
COV_tot = np.empty((len(bold_cl_vstack), len(bold_cl_vstack)), dtype=object)

for i in range(len(bold_cl_vstack)):
    for j in range(len(bold_cl_vstack)):
        ii = tracer1[i]*num_zbins_hi + zz_bin1[i]
        jj = tracer1[j]*num_zbins_hi + zz_bin1[j] 
        c11 = matr_tot[ii, jj] if ii <= jj else matr_tot[jj, ii]

        ii = tracer2[i]*num_zbins_hi + zz_bin2[i]
        jj = tracer2[j]*num_zbins_hi + zz_bin2[j]
        c22 = matr_tot[ii, jj] if ii <= jj else matr_tot[jj, ii]
        
        ii = tracer1[i]*num_zbins_hi + zz_bin1[i]
        jj = tracer2[j]*num_zbins_hi + zz_bin2[j] 
        c12 = matr_tot[ii, jj] if ii <= jj else matr_tot[jj, ii]
        
        ii = tracer2[i]*num_zbins_hi + zz_bin2[i]
        jj = tracer1[j]*num_zbins_hi + zz_bin1[j]
        c21 = matr_tot[ii, jj] if ii <= jj else matr_tot[jj, ii]

        COV_tot[i,j] = ((c11*c22) + (c12*c21)) / ((2*ell + 1) * f_sky)

In [25]:
# 4. Invert Covariance Matrix per \ell mode
inv_COV_tot = []
for l_idx in range(len(ell)):
    cov_i = np.array([[cell[l_idx] for cell in row] for row in COV_tot], dtype=float)
    inv_COV_tot.append(np.linalg.inv(cov_i))

print("Theoretical Covariance Matrix and Inverses built successfully.")

Theoretical Covariance Matrix and Inverses built successfully.


### 8. Noise Computation

In [26]:
# 1. Experimental Configurations
hi_params = {
    'T_sys': 28.0,
    'B': 20e6,
    't_obs': 1.8e7,       # 1 year
    'N_d': 254.0,
    'S_area': 20000.0,
    'n_pol': 1,
    'Area_e': 140,
    'K_fg': 6e-7,
    'A_fg': 0.129,
    'b_fg': -0.081,
    'c_fg': 0.581
}

In [27]:
# 2. Fetch the damping for the currently active network
ell2_damp_gw = gw_config[active_network]['gw_damping']

In [28]:
# 3. Compute Beams 
theta_b = 1.22 * 0.21 * (1 + zz_hi) / 15.0
S_area_rads = hi_params['S_area'] * (np.pi / 180)**2

beam_hi = np.exp(-ell * (ell + 1) * ((theta_b[:, None] / np.sqrt(16 * np.log(2)))**2))
beam_gw = np.exp(-ell * (ell + 1) / ell2_damp_gw[:, None])

In [29]:
# 4. Compute HI Noise (Foreground + Instrumental)
Tb_hi = get_hi_dndz(zz_hi)

Cl_noise_fg = (hi_params['K_fg'] * hi_params['A_fg'] * np.exp(hi_params['b_fg'] * (ell**hi_params['c_fg'])) / f_sky)

Cl_noise_instr = np.zeros((len(zz_hi), len(ell)), dtype=float)
for i in range(len(zz_hi)):
    instr_factor = (hi_params['T_sys'] / Tb_hi[i] / np.sqrt(hi_params['n_pol'] * hi_params['B'] * hi_params['t_obs'] * hi_params['N_d']))
    Cl_noise_instr[i, :] = (theta_b[i]**2) * ((instr_factor * np.sqrt(S_area_rads / theta_b[i] / theta_b[i]))**2)

##### 8.1 Applying Noise to spectra

In [30]:
# 1. HIxHI Observed (Add Foregrounds and Instrumental Noise)
cl_hh_obs = []

idx = 0
for i in range(len(zz_hi)):
    for j in range(i, len(zz_hi)):
        noise = Cl_noise_fg + Cl_noise_instr[i] if j == i else Cl_noise_fg
        cl_hh_obs.append((beam_hi[i] * beam_hi[j] * cl_hh_stack[idx]) + noise)
        idx += 1

cl_hh_obs = np.vstack(cl_hh_obs)

In [31]:
# 2. GWxHI Observed (Beams only, no extra noise terms)
cl_hg_obs = []

idx = 0
for i in range(len(zz_hi)):
    for j in range(len(zz_gw)):
        cl_hg_obs.append(beam_hi[i] * beam_gw[j] * cl_cross_final[idx])
        idx += 1

cl_hg_obs = np.vstack(cl_hg_obs)

In [32]:
# 3. GWxGW Observed (Add Poisson Shot Noise to diagonal)
cl_gg_obs = []

idx = 0
for i in range(len(zz_gw)):
    for j in range(i, len(zz_gw)):
        shot_noise = (1.0 / mean_n_per_bin[i]) if j == i else 0.0
        cl_gg_obs.append((beam_gw[i] * beam_gw[j] * cl_gg_stack[idx]) + shot_noise)
        idx += 1

cl_gg_obs = np.vstack(cl_gg_obs)

bold_cl_obs = np.concatenate((cl_hh_obs, cl_hg_obs, cl_gg_obs))
print(f"Generated Observed Spectra. Total Shape: {bold_cl_obs.shape}")

Generated Observed Spectra. Total Shape: (820, 199)


### 9. Compute Simulated (Inverse) Covariance Matrix

In [33]:
idx = 0

tracer1_err = []
tracer2_err = []
zz_bin1_err = []
zz_bin2_err = []

matr_tot_err = np.empty((num_zbins_tot, num_zbins_tot), dtype=object)

for i in range(len(zz_hi)):
    for j in range(i, len(zz_hi)):
        tracer1_err.append( 0 )
        tracer2_err.append( 0 )
        zz_bin1_err.append( i )
        zz_bin2_err.append( j )
        matr_tot_err[i,j] = cl_hh_obs[idx]
        idx += 1

In [34]:
idx = 0

for i in range(len(zz_hi)):
    for j in range(len(zz_gw)):
        tracer1_err.append( 0 ) 
        tracer2_err.append( 1 ) 
        zz_bin1_err.append( i )
        zz_bin2_err.append( j )
        matr_tot_err[i,j+num_zbins_hi] = cl_hg_obs[idx]
        idx += 1

In [35]:
idx = 0

for i in range(len(zz_gw)):
    for j in range(i, len(zz_gw)):
        tracer1_err.append( 1 )
        tracer2_err.append( 1 ) 
        zz_bin1_err.append( i )
        zz_bin2_err.append( j )
        #matr_hg[i,j] = bold_cl[-1]
        matr_tot_err[i+num_zbins_hi,j+num_zbins_hi] = cl_gg_obs[idx]
        idx += 1

In [36]:
COV_tot_err = np.empty((len(bold_cl_obs), len(bold_cl_obs)), dtype=object)

for i in range(len(bold_cl_obs)):
    for j in range(len(bold_cl_obs)):
        ii = tracer1_err[i]*num_zbins_hi + zz_bin1_err[i]
        jj = tracer1_err[j]*num_zbins_hi + zz_bin1_err[j] 
        c11 = matr_tot_err[ii, jj] if ii <= jj else matr_tot_err[jj, ii]

        ii = tracer2_err[i]*num_zbins_hi + zz_bin2_err[i]
        jj = tracer2_err[j]*num_zbins_hi + zz_bin2_err[j]
        c22 = matr_tot_err[ii, jj] if ii <= jj else matr_tot_err[jj, ii]
        
        ii = tracer1_err[i]*num_zbins_hi + zz_bin1_err[i]
        jj = tracer2_err[j]*num_zbins_hi + zz_bin2_err[j] 
        c12 = matr_tot_err[ii, jj] if ii <= jj else matr_tot_err[jj, ii]
        
        ii = tracer2_err[i]*num_zbins_hi + zz_bin2_err[i]
        jj = tracer1_err[j]*num_zbins_hi + zz_bin1_err[j]
        c21 = matr_tot_err[ii, jj] if ii <= jj else matr_tot_err[jj, ii]

        COV_tot_err[i,j] = ((c11*c22) + (c12*c21)) / ((2*ell + 1)*f_sky)

In [37]:
inv_COV_tot_err = []

for i in range(len(ell)):
    cov_i = np.array([[cell[i] for cell in row] for row in COV_tot_err])
    inv_cov_i = np.linalg.inv(cov_i)

    inv_COV_tot_err.append(inv_cov_i)

##### 9.1 Compute also the matrices for the single contributions

In [38]:
idx = 0

zz_bin1_hh = []
zz_bin2_hh = []

matr_tot_hh = np.empty((len(zz_hi), len(zz_hi)), dtype=object)

for i in range(len(zz_hi)):
    for j in range(i, len(zz_hi)):
        zz_bin1_hh.append( i )
        zz_bin2_hh.append( j )
        matr_tot_hh[i,j] = cl_hh_obs[idx]
        idx += 1

In [39]:
COV_tot_hh = np.empty((len(cl_hh_obs), len(cl_hh_obs)), dtype=object)

for i in range(len(cl_hh_obs)):
    for j in range(len(cl_hh_obs)):
        ii = zz_bin1_hh[i]
        jj = zz_bin1_hh[j] 
        c11 = matr_tot_hh[ii, jj] if ii <= jj else matr_tot_hh[jj, ii]

        ii = zz_bin2_hh[i]
        jj = zz_bin2_hh[j]
        c22 = matr_tot_hh[ii, jj] if ii <= jj else matr_tot_hh[jj, ii]
        
        ii = zz_bin1_hh[i]
        jj = zz_bin2_hh[j] 
        c12 = matr_tot_hh[ii, jj] if ii <= jj else matr_tot_hh[jj, ii]
        
        ii = zz_bin2_hh[i]
        jj = zz_bin1_hh[j]
        c21 = matr_tot_hh[ii, jj] if ii <= jj else matr_tot_hh[jj, ii]

        COV_tot_hh[i,j] = ((c11*c22) + (c12*c21)) / ((2*ell + 1)*f_sky)

In [40]:
inv_COV_tot_hh = []

for i in range(len(ell)):
    cov_i = np.array([[cell[i] for cell in row] for row in COV_tot_hh])
    inv_cov_i = np.linalg.inv(cov_i)

    inv_COV_tot_hh.append(inv_cov_i)

In [41]:
idx = 0

zz_bin1_gg = []
zz_bin2_gg = []

matr_tot_gg = np.empty((len(zz_gw), len(zz_gw)), dtype=object)

for i in range(len(zz_gw)):
    for j in range(i, len(zz_gw)):
        zz_bin1_gg.append( i )
        zz_bin2_gg.append( j )
        matr_tot_gg[i,j] = cl_gg_obs[idx]
        idx += 1

In [42]:
COV_tot_gg = np.empty((len(cl_gg_obs), len(cl_gg_obs)), dtype=object)

for i in range(len(cl_gg_obs)):
    for j in range(len(cl_gg_obs)):
        ii = zz_bin1_gg[i]
        jj = zz_bin1_gg[j] 
        c11 = matr_tot_gg[ii, jj] if ii <= jj else matr_tot_gg[jj, ii]

        ii = zz_bin2_gg[i]
        jj = zz_bin2_gg[j]
        c22 = matr_tot_gg[ii, jj] if ii <= jj else matr_tot_gg[jj, ii]
        
        ii = zz_bin1_gg[i]
        jj = zz_bin2_gg[j] 
        c12 = matr_tot_gg[ii, jj] if ii <= jj else matr_tot_gg[jj, ii]
        
        ii = zz_bin2_gg[i]
        jj = zz_bin1_gg[j]
        c21 = matr_tot_gg[ii, jj] if ii <= jj else matr_tot_gg[jj, ii]

        COV_tot_gg[i,j] = ((c11*c22) + (c12*c21)) / ((2*ell + 1)*f_sky)

In [43]:
inv_COV_tot_gg = []

for i in range(len(ell)):
    cov_i = np.array([[cell[i] for cell in row] for row in COV_tot_gg])
    inv_cov_i = np.linalg.inv(cov_i)

    inv_COV_tot_gg.append(inv_cov_i)

### 10. Export Data to File

In [44]:
# Define output directory and prefix 
out_dir = '/home/matteo/montepython_public-3.6/data/GWxHI/'
prefix = f'bias_scelfoBased_allzbin_{active_network}_trial'

os.makedirs(out_dir, exist_ok=True)

In [45]:
# Define everything to save
export_manifest = {
    # Full Matrix
    f'dataset_GWxHI_{prefix}_Cl.pkl': (bold_cl_obs,), #bold_cl_vstack
    f'dataset_GWxHI_{prefix}_Cov.pkl': (COV_tot_err,), #COV_tot
    f'dataset_GWxHI_{prefix}_InvCov.pkl': (inv_COV_tot_err,), #inv_COV_tot
    f'dataset_GWxHI_{prefix}_zbinhi.pkl': (zz_hi,),
    f'dataset_GWxHI_{prefix}_zbingw.pkl': (zz_gw,),
    f'dataset_GWxHI_{prefix}_zbin.pkl': (z_sorted,),
    f'dataset_GWxHI_{prefix}_zedgeshi.pkl': (zz_edges_hi,),
    f'dataset_GWxHI_{prefix}_zedgesgw.pkl': (zz_edges_eq_gw,),
    f'dataset_GWxHI_{prefix}_NgwBin.pkl': (mean_n_per_bin,),
    
    # GW Only
    f'dataset_GWonly_{prefix}_Cl.pkl': (cl_gg_obs,),
    f'dataset_GWonly_{prefix}_Cov.pkl': (COV_tot_gg,),
    f'dataset_GWonly_{prefix}_InvCov.pkl': (inv_COV_tot_gg,),
    f'dataset_GWonly_{prefix}_zbinhi.pkl': (zz_hi,),
    f'dataset_GWonly_{prefix}_zbingw.pkl': (zz_gw,),
    f'dataset_GWonly_{prefix}_zbin.pkl': (z_sorted,),
    f'dataset_GWonly_{prefix}_zedgeshi.pkl': (zz_edges_hi,),
    f'dataset_GWonly_{prefix}_zedgesgw.pkl': (zz_edges_eq_gw,),
    f'dataset_GWonly_{prefix}_NgwBin.pkl': (mean_n_per_bin,),
    
    # HI Only
    f'dataset_HIonly_{prefix}_Cl.pkl': (cl_hh_obs,),
    f'dataset_HIonly_{prefix}_Cov.pkl': (COV_tot_hh,),
    f'dataset_HIonly_{prefix}_InvCov.pkl': (inv_COV_tot_hh,),
    f'dataset_HIonly_{prefix}_zbinhi.pkl': (zz_hi,),
    f'dataset_HIonly_{prefix}_zbingw.pkl': (zz_gw,),
    f'dataset_HIonly_{prefix}_zbin.pkl': (z_sorted,),
    f'dataset_HIonly_{prefix}_zedgeshi.pkl': (zz_edges_hi,),
    f'dataset_HIonly_{prefix}_zedgesgw.pkl': (zz_edges_eq_gw,),
    f'dataset_HIonly_{prefix}_NgwBin.pkl': (mean_n_per_bin,)
}

print(f"Exporting files to {out_dir}...")

for filename, variables in export_manifest.items():
    filepath = os.path.join(out_dir, filename)
    with open(filepath, 'wb') as f:
        for var in variables:
            dill.dump(var, f)

print("Data generation and export complete!")

Exporting files to /home/matteo/montepython_public-3.6/data/GWxHI/...
Data generation and export complete!


In [46]:
mean_n_per_bin

array([8488.63355414, 8487.93857638, 8485.9318611 , 8487.41394672,
       8485.43541636, 8490.72330132, 8485.34517561, 8488.76940168,
       8488.89210363, 8490.06667991])